# Semana 1: El Lenguaje de la IA - Ejercicios con Dataset Real
## Notebook de Ejercicios (NB2) - Dataset Iris

### Propósito de la sesión
Aplicar los conceptos de representación de datos, funciones y visualización aprendidos en el notebook conceptual a un problema real de clasificación. Trabajaremos con el clásico dataset Iris para explorar sus características, visualizar distribuciones y experimentar manualmente con la separación de clases usando funciones lineales.

### Objetivos de aprendizaje
* Cargar y explorar un dataset real (Iris) usando herramientas de Python.
* Comprender la estructura de una matriz de características (features) y un vector de etiquetas (target).
* Visualizar la distribución de datos univariantes (histogramas) y multivariantes (scatter plots).
* Entender el concepto de separabilidad lineal.
* Ajustar manualmente, mediante ensayo y error, una función lineal para separar dos clases y visualizar la frontera de decisión.

## Configuración Inicial

Importamos las librerías necesarias y cargamos el dataset Iris directamente desde `sklearn`.

In [ ]:
# Importamos las librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Para cargar el dataset Iris
from sklearn.datasets import load_iris

# Configuración de estilo para gráficos
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Librerías importadas correctamente.")

# Cargamos el dataset Iris
iris = load_iris()
print("\nDataset Iris cargado exitosamente desde sklearn.datasets.")

---
## Actividad 1: Cargar el dataset y explorar sus dimensiones

El dataset Iris contiene 150 muestras de flores, con 4 características (features) y 3 especies objetivo (target).

**Características:**
* `sepal length (cm)` - Longitud del sépalo
* `sepal width (cm)` - Ancho del sépalo
* `petal length (cm)` - Longitud del pétalo
* `petal width (cm)` - Ancho del pétalo

**Especies:**
* 0: Setosa
* 1: Versicolor
* 2: Virginica

In [ ]:
# Exploramos la estructura del dataset
print("Claves del dataset:", iris.keys())

# Matriz de características (X) - un tensor 2D (matriz)
X = iris.data
print(f"\nMatriz de características (X):")
print(f"  - Dimensiones: {X.shape}")
print(f"  - Tipo de datos: {X.dtype}")
print(f"  - Primeras 5 filas:\n{X[:5]}")

# Vector de etiquetas (y)
y = iris.target
print(f"\nVector de etiquetas (y):")
print(f"  - Dimensiones: {y.shape}")
print(f"  - Primeras 5 etiquetas: {y[:5]}")
print(f"  - Clases únicas: {np.unique(y)}")

# Nombres de las características y las clases
feature_names = iris.feature_names
target_names = iris.target_names
print(f"\nNombres de características: {feature_names}")
print(f"Nombres de clases: {target_names}")

# Creamos un DataFrame de pandas para facilitar la manipulación
df = pd.DataFrame(X, columns=feature_names)
df['species'] = pd.Categorical.from_codes(y, target_names)
print("\nPrimeras 5 filas del DataFrame:")
print(df.head())

### Reflexión
La matriz de características $X$ es un tensor de dimensiones $150 \times 4$. Cada fila es un **vector** que representa una flor. Cada columna es un **escalar** (una característica). Las etiquetas $y$ son un **vector** de 150 elementos. ¡Estamos viendo las estructuras de datos del NB1 en acción!

---
## Actividad 2: Visualizar las distribuciones de cada característica usando histogramas

Los histogramas nos permiten ver cómo se distribuyen los valores de cada característica y si estas distribuciones pueden ayudar a distinguir las especies.

In [ ]:
# Configuramos la figura para los histogramas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()  # Aplanamos para iterar fácilmente

# Colores para las especies
colors = ['blue', 'green', 'red']

# Iteramos sobre cada característica
for idx, feature in enumerate(feature_names):
    for species_idx, species_name in enumerate(target_names):
        subset = df[df['species'] == species_name]
        axes[idx].hist(subset[feature], bins=15, alpha=0.7,
                       label=species_name, color=colors[species_idx], edgecolor='black')
    axes[idx].set_title(f'Distribución de {feature}')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Histogramas de las Características del Dataset Iris', y=1.02, fontsize=16)
plt.show()

### Análisis de los histogramas
Observa cómo la especie Setosa (azul) es claramente separable de las otras dos usando las características de pétalo (longitud y ancho). Sin embargo, Versicolor y Virginica se superponen parcialmente. ¿Qué característica crees que podría separar mejor a Versicolor de Virginica?

In [ ]:
# Visualización adicional: Pairplot (matriz de scatter plots)
# Muestra las relaciones entre pares de características
print("Generando pairplot para visualizar relaciones multivariantes...")
sns.pairplot(df, hue='species', palette='husl', diag_kind='hist')
plt.suptitle('Pairplot de Características del Iris', y=1.02, fontsize=16)
plt.show()

### Reflexión sobre el pairplot
En los gráficos de dispersión (scatter plots) podemos ver la relación entre dos características. Observa cómo en el gráfico de 'petal length' vs 'petal width', los puntos de Setosa forman un grupo claramente aislado. Este tipo de visualización nos da una pista de que podríamos separar Setosa del resto con una simple **línea recta**. Esa es la idea de la **separabilidad lineal**.

---
## Actividad 3: Ajuste manual de una función lineal para separar dos clases

Basado en el pairplot, elegimos trabajar con las características 'petal length' y 'petal width' para separar la clase **Setosa** de las clases **Versicolor y Virginica** (que trataremos como una sola clase "No Setosa").

Una **función lineal** en 2D tiene la forma:
$$f(x_1, x_2) = w_1 x_1 + w_2 x_2 + b = 0$$

Donde:
* $x_1$: longitud del pétalo
* $x_2$: ancho del pétalo
* $w_1$, $w_2$: pesos (pendientes)
* $b$: sesgo (intercepto)

La **frontera de decisión** es la línea donde $f(x_1, x_2) = 0$. Los puntos con $f(x_1, x_2) > 0$ se clasificarán como una clase, y los que tengan $f(x_1, x_2) < 0$ como la otra.

**Actividad interactiva:** Modifica los valores de `w1`, `w2` y `b` en la celda de abajo para encontrar una línea que separe los puntos azules (Setosa) de los puntos verdes y rojos (No Setosa).

In [ ]:
# Preparamos los datos para la clasificación binaria (Setosa vs Resto)
# Convertimos las etiquetas: 1 para Setosa, 0 para las demás
y_bin = (y == 0).astype(int)  # 1 si es Setosa, 0 si es Versicolor/Virginica

# Usamos solo las características de pétalo (índices 2 y 3)
X_petal = X[:, 2:4]  # petal length, petal width

# Visualizamos los datos
plt.figure(figsize=(10, 6))
plt.scatter(X_petal[y_bin == 1, 0], X_petal[y_bin == 1, 1],
            color='blue', label='Setosa', edgecolor='k', s=100, alpha=0.7)
plt.scatter(X_petal[y_bin == 0, 0], X_petal[y_bin == 0, 1],
            color='gray', label='No Setosa', edgecolor='k', s=100, alpha=0.5)

# --- PARÁMETROS EDITABLES PARA LA LÍNEA DE SEPARACIÓN ---
w1 = 1.0   # Peso para petal length
w2 = -2.0  # Peso para petal width
b = 0.5    # Sesgo
# ---------------------------------------------------------

# Generamos puntos para dibujar la línea w1*x1 + w2*x2 + b = 0
# Despejamos x2: x2 = (-w1*x1 - b) / w2
x1_line = np.linspace(0, 7, 100)
if w2 != 0:
    x2_line = (-w1 * x1_line - b) / w2
    plt.plot(x1_line, x2_line, 'r--', linewidth=2,
             label=f'Frontera: {w1:.1f}*Largo + {w2:.1f}*Ancho + {b:.1f} = 0')
else:
    # Si w2 es 0, la línea es vertical: x1 = -b/w1
    x1_vert = -b / w1
    plt.axvline(x=x1_vert, color='r', linestyle='--', linewidth=2,
                label=f'Frontera: {w1:.1f}*Largo + {b:.1f} = 0')

plt.xlabel('Petal length (cm)', fontsize=14)
plt.ylabel('Petal width (cm)', fontsize=14)
plt.title('Ajuste Manual de una Frontera Lineal para Separar Setosa', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xlim(0, 7.5)
plt.ylim(0, 3)
plt.show()

# Evaluamos la precisión de esta línea
predicciones = (w1 * X_petal[:, 0] + w2 * X_petal[:, 1] + b) > 0
accuracy = np.mean(predicciones == y_bin)
print(f"\nPrecisión de la clasificación con los pesos actuales: {accuracy * 100:.2f}%")

# Mostramos algunos errores de clasificación
errores = np.where(predicciones != y_bin)[0]
if len(errores) > 0:
    print(f"Número de puntos mal clasificados: {len(errores)}")
    print("Índices de los puntos mal clasificados:", errores)
else:
    print("¡Clasificación perfecta con esta línea!")

### Instrucciones para el experimento manual
1. **Ejecuta la celda** con los valores iniciales `w1=1.0, w2=-2.0, b=0.5`.
2. Observa la línea roja discontinua y la precisión obtenida.
3. **Modifica los valores** de `w1`, `w2` y `b` en la celda (la sección marcada como PARÁMETROS EDITABLES).
4. Vuelve a ejecutar la celda.
5. Intenta encontrar la combinación que maximice la precisión (que clasifique correctamente todos los puntos).

**Preguntas para reflexionar:**
*   ¿Qué combinación de pesos te dio la mejor separación?
*   ¿Es posible separar perfectamente Setosa del resto con una línea recta? ¿Por qué?
*   ¿Crees que sería igual de fácil separar Versicolor de Virginica con una línea recta? ¿Por qué (basado en los pairplots)?

In [ ]:
# BONUS: Visualización 3D de la función lineal
# Para entender mejor la función f(x1, x2) = w1*x1 + w2*x2 + b

from mpl_toolkits.mplot3d import Axes3D

# Tomamos los pesos de la celda anterior (o usa los que encontraste)
# Para este ejemplo, usaremos una línea que funciona bien (puedes cambiarlos)
w1_best, w2_best, b_best = 1.0, -2.5, 1.5

# Creamos una malla de puntos para el plano
x1_surf, x2_surf = np.meshgrid(np.linspace(0, 7, 30), np.linspace(0, 3, 30))
f_surf = w1_best * x1_surf + w2_best * x2_surf + b_best

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

# Puntos de datos
ax.scatter(X_petal[y_bin==1, 0], X_petal[y_bin==1, 1], np.zeros(np.sum(y_bin==1)),
           c='blue', s=50, label='Setosa (y=1)', alpha=0.8, edgecolor='k')
ax.scatter(X_petal[y_bin==0, 0], X_petal[y_bin==0, 1], np.zeros(np.sum(y_bin==0)),
           c='gray', s=50, label='No Setosa (y=0)', alpha=0.5, edgecolor='k')

# Superficie del plano
ax.plot_surface(x1_surf, x2_surf, f_surf, alpha=0.3, cmap='coolwarm')

# Línea donde f=0 (frontera de decisión)
ax.contour(x1_surf, x2_surf, f_surf, levels=[0], colors='red', linewidths=3)

ax.set_xlabel('Petal length (cm)')
ax.set_ylabel('Petal width (cm)')
ax.set_zlabel('f(x1, x2)')
ax.set_title('Visualización 3D de la Función Lineal y Frontera de Decisión')
ax.legend()
plt.show()

print("El plano es la función f(x1, x2). La frontera de decisión es la línea roja donde el plano corta el nivel f=0.")
print("Los puntos azules (Setosa) están por encima de este plano (f>0) y los grises por debajo (f<0).")

---
## Ejercicios para el Estudiante

1.  **Exploración de Características:** Repite la Actividad 3 (ajuste manual de la frontera), pero esta vez usa las características 'sepal length' y 'sepal width'. ¿Es tan fácil separar Setosa del resto usando estas características? ¿Por qué crees que sucede esto?

2.  **Análisis de Error:** En el ajuste manual que hiciste con 'petal length' y 'petal width', ¿hubo algún punto mal clasificado? Si es así, inspecciona ese punto en el DataFrame (`df.iloc[índice_del_error]`). ¿A qué especie pertenece realmente? ¿Por qué crees que tu línea lo clasificó mal?

3.  **Desafío de la No-Linealidad:** (Reflexión) ¿Qué tipo de función crees que se necesitaría para separar perfectamente las tres clases a la vez? ¿Por qué una sola línea recta no es suficiente?

---
## Conclusiones de la Sesión

*   Hemos trabajado con un **dataset real (Iris)**, confirmando que los datos se organizan en matrices (características) y vectores (etiquetas).
*   Las **visualizaciones (histogramas y pairplots)** son herramientas fundamentales para entender la distribución de los datos y la relación entre características.
*   El concepto de **separabilidad lineal** es clave: algunas clases (Setosa) pueden separarse de otras con una simple línea recta, mientras que otras (Versicolor y Virginica) presentan un desafío mayor.
*   Hemos experimentado de forma práctica con una **función lineal** (modelo de clasificación) y su **frontera de decisión**, ajustando sus parámetros manualmente para comprender su efecto.
*   Este ejercicio manual sienta las bases para entender cómo los algoritmos de Machine Learning (como la regresión logística o las SVM) aprenden automáticamente estos parámetros para maximizar la precisión.

¡Has dado el primer paso práctico en la construcción de modelos de clasificación!